<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c03-header.png" alt="Nextia Learning · Essential Mathematics and Statistics for AI" width="100%">

# Samples and populations

**[Samples and populations](https://learning.nextia-ai.com/courses/math/m05/samples-and-populations/)** · Essential Mathematics and Statistics for AI · Module 5, lesson 1 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**Optional: check the numbers in Python.** This notebook is optional. The lesson works without code: a calculator is enough. Use the notebook to check the lesson's numbers and to redo its worked examples and tasks in Python.

**You will** draw random samples and rated-only samples from a known population of 6,000 reply times, and see that more data does not remove sampling bias. Then see why linked tickets make a result less certain, and split tickets by customer to prevent data leakage.

| | |
|---|---|
| **Time** | About 20 minutes, after you read the lesson |
| **Needs** | Nothing to install and no account: only Python's standard library. It runs in Colab, Kaggle or any Jupyter. |
| **You should know** | The mean, from [Describe a dataset: The mean and the median](https://learning.nextia-ai.com/courses/math/m01/describe-a-dataset/#the-mean-and-the-median), and how to read a histogram, from [Read distributions: One dataset, three views](https://learning.nextia-ai.com/courses/math/m01/read-distributions/#one-dataset-three-views). |
| **Tested** | 2026-10-07, Python 3.14.6 (standard library only), in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

The [lesson page](https://learning.nextia-ai.com/courses/math/m05/samples-and-populations/) has the full explanations, the charts, the explorers and the knowledge checks that count toward your certificate. Read a section there, then check its numbers here.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, replace each `...` with your calculation, run the cell, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C03/M05-L01-samples-and-populations/samples-and-populations-solution.ipynb).

## Setup: the helpers

The next cell defines three small functions that the notebook uses. You do not need to read them, but you can.

- `expect(what, yours, expected)` compares your number with the lesson's number. It says "Check passed." or if yours is too high or too low. It does not show the answer.
- `table(headers, rows)` prints a small table with aligned columns.
- `bars(labels, counts)` prints a text bar chart: one row of `█` for each label.

Run the cell. You should see `Ready.`

In [ ]:
# Helpers: check an answer without showing it, print a table, draw text bars.
import math, random, statistics

def expect(what, yours, expected, tol=0.01):
    """Compare your value with the expected one, to within `tol` for numbers."""
    if yours is ...:
        print(f"Not yet: replace ... with your calculation for {what}, then run the cell again.")
        return
    if isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        if abs(yours - expected) <= tol:
            print(f"Check passed: {what} is {yours:g}.")
        else:
            side = "too high" if yours > expected else "too low"
            print(f"Not yet: your {what}, {yours:g}, is {side}. Look at the step before it again.")
    elif yours == expected:
        print(f"Check passed: {what} is {yours}.")
    else:
        print(f"Not yet: your {what} is {yours!r}. That is not the expected answer.")

def table(headers, rows, digits=2):
    """Print rows as a table. Floats are rounded to `digits` decimals."""
    cell = lambda v: f"{v:,.{digits}f}" if isinstance(v, float) else str(v)
    text = [[cell(v) for v in row] for row in rows]
    widths = [max(len(str(h)), *(len(r[i]) for r in text)) for i, h in enumerate(headers)]
    print("  ".join(str(h).rjust(w) for h, w in zip(headers, widths)))
    for r in text:
        print("  ".join(v.rjust(w) for v, w in zip(r, widths)))

def bars(labels, counts, width=40):
    """Print a text bar chart: the longest bar is `width` characters."""
    top = max(counts) or 1
    size = max(len(str(label)) for label in labels)
    for label, n in zip(labels, counts):
        print(f"{str(label).rjust(size)} | {'█' * round(n / top * width)} {n}")

print("Ready.")

## Setup: the population of 6,000 reply times

The next cell defines `population`: the first-reply times, in minutes, of 6,000 made-up past tickets (about five weeks). It is the same population as in the lesson's explorer. Because you have all of it, you know the true mean.

Run the cell. You should see `Ready: 6000 tickets, true mean 26.8 minutes`.

In [ ]:
# The data, written in full (you do not need to read it).
population = [
    24, 9, 11, 32, 15, 48, 26, 33, 20, 11, 43, 44, 20, 11, 56, 14, 33, 60, 15, 20,
    24, 28, 14, 42, 22, 34, 35, 10, 13, 24, 31, 35, 15, 19, 18, 38, 14, 6, 27, 17,
    14, 14, 23, 8, 12, 26, 76, 20, 19, 15, 29, 40, 25, 38, 27, 51, 14, 30, 17, 15,
    24, 27, 90, 15, 23, 11, 39, 24, 24, 130, 13, 6, 8, 13, 21, 29, 20, 66, 12, 21,
    18, 14, 11, 10, 27, 30, 32, 13, 46, 38, 60, 9, 18, 51, 10, 11, 28, 16, 7, 26,
    26, 35, 72, 24, 15, 13, 20, 42, 92, 17, 62, 37, 13, 11, 6, 13, 12, 24, 22, 70,
    10, 6, 13, 15, 23, 24, 11, 11, 73, 27, 93, 44, 19, 45, 10, 19, 73, 5, 16, 29,
    30, 18, 47, 13, 58, 46, 10, 13, 20, 17, 19, 41, 10, 19, 16, 52, 19, 35, 33, 14,
    57, 42, 14, 29, 18, 15, 29, 29, 11, 23, 17, 13, 65, 12, 25, 18, 16, 9, 41, 8,
    23, 25, 51, 22, 14, 25, 30, 24, 31, 21, 35, 8, 55, 24, 22, 10, 10, 30, 26, 17,
    27, 38, 43, 14, 111, 9, 14, 41, 12, 42, 22, 32, 28, 38, 37, 26, 18, 29, 19, 12,
    18, 64, 23, 23, 27, 9, 31, 10, 20, 42, 19, 11, 81, 17, 13, 11, 18, 51, 10, 18,
    15, 11, 12, 30, 6, 47, 27, 21, 16, 15, 27, 34, 41, 78, 20, 61, 12, 29, 19, 56,
    24, 68, 28, 20, 51, 13, 24, 13, 18, 30, 48, 18, 19, 9, 31, 27, 17, 32, 21, 14,
    23, 13, 29, 52, 40, 60, 16, 41, 37, 118, 20, 17, 37, 24, 46, 69, 13, 14, 38, 30,
    10, 32, 12, 13, 13, 36, 18, 15, 11, 23, 9, 22, 29, 8, 17, 6, 13, 10, 10, 26,
    22, 24, 27, 18, 9, 10, 16, 32, 24, 93, 23, 22, 29, 22, 20, 28, 13, 31, 40, 23,
    15, 56, 81, 58, 19, 18, 33, 12, 42, 22, 12, 67, 20, 21, 9, 26, 72, 34, 46, 11,
    25, 28, 26, 19, 19, 20, 33, 9, 27, 18, 45, 21, 10, 15, 7, 56, 7, 51, 129, 26,
    15, 16, 24, 7, 15, 21, 48, 61, 5, 29, 40, 66, 23, 62, 19, 11, 27, 53, 28, 13,
    9, 11, 104, 24, 12, 18, 18, 20, 10, 27, 16, 11, 10, 37, 8, 21, 18, 165, 27, 34,
    40, 17, 29, 57, 13, 45, 20, 18, 26, 19, 30, 28, 17, 59, 32, 30, 52, 29, 31, 17,
    42, 41, 18, 16, 17, 93, 8, 25, 33, 17, 29, 40, 47, 28, 23, 33, 13, 30, 37, 25,
    42, 23, 10, 54, 25, 5, 31, 30, 10, 56, 12, 52, 48, 26, 51, 113, 20, 60, 13, 29,
    20, 55, 15, 14, 16, 40, 11, 14, 23, 26, 26, 15, 60, 71, 15, 18, 10, 21, 11, 51,
    28, 12, 23, 32, 17, 16, 26, 25, 48, 35, 38, 49, 63, 12, 7, 12, 32, 55, 19, 9,
    15, 75, 17, 13, 6, 25, 12, 5, 22, 34, 47, 25, 41, 37, 6, 26, 17, 106, 5, 25,
    18, 34, 37, 31, 17, 16, 33, 31, 20, 48, 53, 29, 7, 15, 23, 33, 24, 23, 13, 25,
    42, 9, 8, 9, 7, 43, 31, 14, 32, 85, 19, 25, 16, 35, 65, 10, 15, 32, 11, 12,
    12, 27, 30, 31, 23, 21, 27, 33, 36, 16, 75, 14, 21, 21, 20, 32, 65, 15, 31, 18,
    11, 34, 15, 54, 12, 37, 16, 30, 18, 25, 59, 23, 16, 13, 12, 70, 24, 29, 18, 28,
    9, 14, 10, 63, 15, 22, 10, 15, 12, 34, 9, 17, 20, 22, 19, 23, 19, 15, 21, 9,
    5, 39, 28, 26, 21, 27, 22, 17, 53, 7, 82, 59, 4, 23, 53, 27, 55, 14, 44, 21,
    25, 23, 32, 29, 34, 42, 32, 40, 63, 14, 34, 15, 17, 20, 21, 6, 17, 26, 48, 26,
    40, 26, 26, 13, 42, 53, 21, 41, 12, 8, 20, 21, 16, 53, 33, 17, 22, 14, 30, 57,
    35, 17, 12, 33, 17, 9, 7, 48, 21, 28, 26, 30, 26, 14, 20, 12, 22, 21, 45, 15,
    40, 17, 38, 13, 26, 26, 23, 12, 3, 26, 15, 8, 19, 18, 34, 16, 26, 9, 43, 20,
    7, 19, 32, 19, 17, 7, 12, 8, 10, 38, 37, 67, 26, 12, 16, 50, 16, 21, 17, 7,
    20, 22, 9, 23, 4, 21, 34, 24, 19, 26, 44, 21, 27, 18, 12, 25, 33, 41, 8, 19,
    33, 19, 22, 59, 29, 57, 39, 10, 12, 13, 12, 8, 12, 23, 16, 32, 15, 45, 34, 29,
    9, 45, 32, 27, 23, 52, 27, 37, 19, 20, 27, 14, 16, 28, 33, 10, 36, 31, 24, 20,
    22, 8, 22, 22, 7, 17, 33, 34, 41, 11, 14, 8, 8, 40, 23, 32, 66, 39, 20, 31,
    37, 24, 21, 18, 18, 17, 31, 50, 9, 61, 22, 58, 13, 16, 14, 4, 29, 34, 13, 22,
    8, 22, 77, 18, 59, 25, 12, 19, 16, 19, 20, 26, 45, 14, 42, 28, 16, 31, 25, 9,
    10, 28, 33, 36, 31, 34, 18, 37, 20, 28, 7, 24, 34, 21, 21, 15, 11, 21, 111, 16,
    8, 16, 27, 38, 15, 10, 31, 3, 15, 81, 6, 46, 17, 6, 7, 8, 51, 17, 36, 14,
    21, 39, 23, 12, 50, 36, 22, 23, 25, 47, 19, 33, 44, 10, 16, 47, 17, 11, 16, 9,
    43, 12, 22, 11, 22, 15, 22, 41, 10, 51, 18, 28, 13, 8, 18, 17, 14, 12, 19, 11,
    6, 17, 13, 23, 63, 18, 24, 16, 15, 32, 48, 25, 15, 27, 17, 22, 42, 29, 22, 6,
    7, 22, 30, 17, 15, 39, 17, 33, 43, 13, 65, 8, 10, 14, 33, 7, 8, 12, 51, 21,
    57, 46, 49, 17, 6, 31, 12, 24, 17, 16, 20, 9, 14, 16, 47, 27, 111, 6, 60, 33,
    17, 20, 23, 12, 36, 20, 94, 21, 11, 9, 19, 9, 39, 24, 22, 50, 51, 18, 15, 45,
    21, 22, 41, 16, 44, 12, 13, 20, 11, 49, 63, 17, 7, 33, 12, 13, 48, 30, 10, 74,
    17, 12, 38, 10, 16, 17, 20, 19, 6, 21, 39, 69, 35, 24, 17, 29, 9, 100, 6, 59,
    25, 29, 7, 26, 37, 11, 18, 29, 53, 36, 9, 17, 14, 26, 29, 16, 36, 15, 20, 16,
    46, 19, 25, 11, 57, 21, 5, 28, 36, 6, 29, 51, 12, 39, 8, 26, 12, 17, 16, 16,
    17, 18, 3, 13, 26, 22, 43, 19, 12, 30, 19, 56, 15, 7, 36, 43, 7, 12, 16, 48,
    34, 93, 22, 29, 12, 25, 14, 26, 58, 18, 29, 20, 10, 12, 18, 6, 13, 38, 21, 18,
    6, 41, 10, 29, 15, 17, 21, 30, 9, 8, 57, 15, 36, 42, 98, 16, 8, 15, 16, 9,
    20, 25, 19, 22, 29, 73, 20, 16, 24, 24, 25, 27, 16, 23, 52, 21, 19, 28, 9, 26,
    30, 23, 17, 55, 46, 10, 9, 13, 22, 18, 33, 15, 17, 55, 18, 36, 37, 12, 9, 38,
    28, 18, 15, 55, 17, 35, 29, 54, 8, 69, 14, 10, 19, 8, 23, 43, 37, 14, 15, 47,
    10, 29, 43, 44, 11, 66, 39, 13, 24, 16, 25, 13, 12, 30, 7, 31, 34, 26, 39, 90,
    15, 21, 77, 38, 23, 13, 25, 24, 31, 72, 25, 16, 17, 33, 6, 49, 14, 25, 9, 50,
    16, 34, 20, 5, 36, 15, 11, 58, 12, 24, 12, 34, 21, 18, 8, 28, 72, 14, 6, 27,
    26, 45, 17, 12, 9, 14, 7, 26, 18, 45, 49, 9, 6, 13, 54, 18, 21, 12, 22, 6,
    102, 36, 15, 28, 15, 23, 11, 7, 57, 31, 97, 20, 22, 22, 15, 12, 13, 33, 52, 35,
    31, 17, 15, 35, 9, 30, 21, 20, 23, 31, 88, 18, 30, 32, 42, 23, 44, 13, 55, 17,
    25, 24, 30, 51, 26, 21, 28, 60, 12, 20, 20, 19, 18, 47, 24, 27, 9, 10, 30, 15,
    48, 29, 59, 26, 20, 20, 42, 65, 10, 23, 38, 19, 11, 19, 56, 10, 37, 19, 52, 42,
    22, 25, 20, 15, 14, 12, 12, 12, 16, 19, 11, 8, 25, 16, 7, 15, 12, 43, 6, 21,
    15, 17, 37, 18, 16, 23, 41, 15, 22, 11, 18, 30, 21, 37, 11, 42, 106, 32, 17, 34,
    21, 16, 35, 23, 20, 26, 21, 24, 15, 31, 65, 38, 19, 41, 56, 48, 29, 20, 17, 52,
    42, 24, 24, 6, 11, 52, 10, 45, 16, 11, 16, 41, 40, 10, 23, 20, 15, 29, 43, 23,
    24, 10, 14, 27, 36, 10, 34, 53, 10, 18, 23, 40, 25, 31, 14, 34, 6, 32, 50, 38,
    32, 24, 32, 12, 17, 26, 41, 31, 9, 9, 45, 22, 52, 9, 41, 13, 13, 77, 26, 49,
    72, 34, 14, 20, 9, 27, 15, 16, 33, 17, 16, 65, 12, 14, 24, 21, 10, 50, 15, 30,
    28, 16, 29, 16, 10, 23, 28, 35, 22, 53, 26, 30, 6, 12, 18, 29, 10, 53, 15, 9,
    47, 6, 27, 14, 13, 38, 35, 21, 45, 8, 7, 15, 12, 20, 45, 8, 15, 29, 27, 52,
    12, 48, 10, 12, 10, 15, 23, 53, 22, 23, 36, 17, 25, 41, 42, 29, 23, 27, 29, 37,
    27, 33, 50, 22, 74, 48, 22, 16, 21, 85, 77, 23, 37, 43, 84, 19, 43, 13, 6, 10,
    11, 21, 37, 35, 18, 35, 43, 18, 18, 30, 19, 58, 20, 11, 30, 44, 15, 44, 10, 63,
    21, 26, 35, 72, 16, 15, 25, 8, 53, 9, 26, 19, 9, 59, 31, 21, 33, 14, 38, 58,
    32, 58, 33, 14, 22, 17, 11, 38, 25, 32, 12, 12, 28, 13, 20, 32, 16, 22, 17, 21,
    6, 21, 49, 17, 22, 79, 24, 34, 37, 42, 14, 27, 28, 28, 32, 41, 52, 17, 33, 47,
    38, 34, 45, 19, 16, 19, 36, 20, 40, 21, 16, 37, 31, 35, 12, 47, 16, 29, 29, 23,
    40, 27, 70, 24, 57, 21, 21, 24, 33, 12, 17, 16, 14, 17, 67, 9, 18, 43, 61, 15,
    8, 32, 24, 22, 6, 26, 9, 9, 32, 26, 61, 29, 13, 16, 43, 13, 19, 20, 27, 22,
    20, 9, 18, 16, 43, 22, 19, 22, 6, 28, 23, 20, 30, 24, 41, 26, 31, 49, 23, 19,
    14, 22, 16, 11, 29, 14, 63, 28, 16, 30, 15, 25, 19, 37, 58, 9, 27, 15, 22, 76,
    25, 15, 7, 18, 35, 18, 20, 18, 15, 13, 51, 23, 47, 20, 39, 36, 13, 29, 13, 26,
    61, 7, 51, 28, 31, 64, 13, 19, 18, 14, 17, 44, 9, 9, 16, 60, 77, 46, 55, 43,
    23, 11, 18, 36, 17, 30, 66, 16, 19, 20, 19, 32, 25, 47, 21, 23, 32, 7, 15, 34,
    34, 20, 10, 53, 30, 15, 21, 51, 12, 46, 15, 40, 62, 15, 36, 42, 10, 21, 24, 21,
    13, 40, 34, 91, 21, 32, 11, 17, 29, 13, 17, 11, 52, 66, 24, 12, 17, 11, 36, 43,
    11, 34, 37, 32, 9, 19, 17, 124, 24, 5, 17, 17, 33, 40, 33, 40, 13, 5, 12, 38,
    12, 19, 49, 21, 20, 9, 11, 13, 17, 10, 42, 23, 31, 11, 42, 66, 40, 20, 35, 20,
    21, 62, 20, 36, 28, 19, 10, 3, 15, 9, 16, 51, 11, 17, 98, 25, 10, 26, 34, 6,
    13, 17, 32, 11, 31, 31, 9, 27, 106, 18, 11, 17, 32, 24, 20, 90, 13, 24, 39, 21,
    16, 8, 16, 11, 48, 22, 37, 20, 21, 27, 11, 48, 7, 40, 16, 11, 59, 6, 25, 19,
    28, 43, 87, 5, 50, 18, 53, 64, 24, 75, 52, 50, 20, 19, 29, 25, 40, 28, 13, 38,
    25, 40, 26, 23, 22, 17, 15, 13, 10, 10, 15, 18, 24, 13, 20, 66, 49, 49, 24, 35,
    41, 15, 16, 9, 21, 15, 8, 26, 40, 36, 23, 24, 17, 38, 25, 18, 19, 45, 32, 17,
    9, 32, 12, 46, 9, 16, 23, 52, 16, 13, 12, 13, 31, 14, 29, 11, 39, 12, 20, 26,
    21, 22, 21, 29, 26, 15, 37, 93, 47, 23, 35, 29, 15, 19, 11, 14, 12, 19, 7, 25,
    10, 27, 32, 17, 18, 19, 7, 31, 5, 11, 10, 9, 30, 23, 6, 21, 37, 55, 26, 23,
    11, 14, 13, 66, 9, 25, 34, 25, 18, 30, 14, 9, 95, 14, 22, 51, 35, 18, 23, 16,
    25, 12, 57, 27, 25, 17, 11, 20, 30, 18, 13, 10, 16, 13, 54, 15, 44, 20, 13, 8,
    45, 16, 12, 39, 30, 92, 33, 31, 15, 6, 12, 43, 47, 16, 11, 21, 23, 44, 7, 72,
    24, 10, 26, 73, 20, 43, 14, 18, 23, 37, 6, 9, 7, 34, 14, 41, 43, 8, 14, 10,
    27, 19, 36, 19, 21, 29, 13, 24, 14, 29, 53, 16, 96, 24, 9, 34, 16, 46, 7, 11,
    24, 22, 13, 11, 23, 28, 30, 21, 70, 31, 25, 19, 22, 8, 13, 65, 29, 44, 48, 21,
    41, 12, 23, 22, 16, 50, 8, 31, 29, 7, 40, 17, 23, 42, 24, 57, 19, 15, 8, 102,
    6, 44, 10, 14, 34, 7, 30, 16, 9, 39, 31, 10, 23, 27, 27, 9, 7, 41, 26, 20,
    38, 40, 29, 9, 13, 20, 11, 12, 40, 33, 18, 24, 17, 16, 14, 35, 36, 17, 23, 15,
    24, 30, 38, 9, 29, 18, 41, 14, 19, 6, 17, 16, 10, 18, 17, 30, 23, 21, 13, 9,
    36, 10, 22, 11, 49, 10, 10, 17, 42, 44, 17, 8, 45, 53, 11, 6, 52, 40, 16, 26,
    20, 40, 47, 48, 22, 9, 27, 42, 24, 62, 15, 29, 13, 23, 13, 35, 37, 45, 23, 32,
    33, 12, 16, 21, 13, 43, 33, 32, 40, 31, 45, 37, 16, 34, 71, 65, 23, 29, 18, 42,
    23, 10, 28, 13, 21, 35, 24, 14, 16, 10, 17, 28, 109, 21, 45, 20, 19, 23, 53, 42,
    9, 28, 15, 19, 22, 21, 56, 15, 14, 13, 56, 21, 26, 8, 27, 21, 12, 24, 12, 28,
    47, 28, 10, 41, 20, 16, 19, 37, 63, 26, 28, 29, 60, 23, 40, 23, 19, 6, 10, 33,
    40, 22, 12, 42, 45, 11, 45, 29, 63, 30, 44, 12, 31, 46, 11, 30, 52, 40, 18, 12,
    21, 17, 55, 25, 17, 34, 53, 13, 14, 18, 61, 18, 8, 19, 20, 14, 17, 56, 21, 17,
    19, 35, 4, 13, 26, 22, 14, 21, 50, 47, 24, 9, 22, 46, 11, 14, 15, 13, 16, 16,
    30, 16, 60, 17, 31, 14, 11, 14, 10, 11, 24, 22, 28, 18, 20, 59, 28, 27, 26, 34,
    19, 9, 21, 10, 17, 17, 67, 21, 51, 29, 46, 14, 24, 24, 23, 17, 9, 11, 12, 36,
    14, 23, 12, 6, 39, 8, 63, 25, 14, 30, 38, 41, 7, 22, 24, 11, 22, 31, 9, 13,
    10, 124, 9, 22, 19, 19, 14, 38, 40, 8, 15, 19, 33, 28, 27, 40, 40, 25, 17, 8,
    28, 31, 21, 19, 25, 21, 23, 67, 23, 15, 19, 12, 18, 18, 20, 23, 20, 24, 45, 13,
    44, 28, 18, 28, 18, 11, 17, 24, 8, 12, 56, 17, 14, 13, 24, 30, 64, 17, 25, 12,
    11, 39, 16, 31, 28, 13, 83, 8, 16, 74, 22, 26, 7, 21, 6, 16, 27, 6, 24, 17,
    36, 22, 12, 7, 19, 29, 39, 95, 16, 34, 12, 10, 18, 13, 16, 17, 49, 22, 27, 25,
    18, 26, 15, 13, 18, 26, 31, 14, 42, 42, 13, 55, 5, 37, 12, 29, 17, 48, 26, 24,
    21, 20, 14, 25, 19, 38, 15, 29, 51, 14, 44, 46, 16, 22, 24, 28, 11, 19, 39, 27,
    8, 26, 10, 25, 15, 26, 15, 17, 8, 29, 16, 10, 17, 20, 22, 39, 22, 67, 13, 12,
    18, 6, 11, 13, 38, 18, 18, 22, 11, 9, 10, 63, 26, 15, 19, 6, 8, 22, 71, 36,
    14, 25, 57, 50, 22, 15, 11, 8, 39, 12, 8, 14, 24, 10, 10, 14, 21, 14, 31, 52,
    47, 5, 24, 13, 11, 28, 14, 6, 8, 5, 39, 19, 43, 24, 16, 19, 21, 18, 7, 43,
    13, 17, 11, 40, 17, 22, 22, 18, 14, 11, 99, 37, 18, 21, 48, 25, 23, 18, 26, 18,
    27, 18, 42, 20, 34, 18, 13, 6, 31, 15, 14, 36, 26, 19, 32, 30, 39, 66, 14, 20,
    17, 48, 17, 35, 38, 10, 61, 11, 13, 29, 48, 21, 14, 16, 6, 18, 8, 22, 50, 28,
    16, 17, 14, 17, 11, 6, 9, 24, 46, 40, 43, 31, 32, 24, 11, 96, 9, 28, 19, 29,
    19, 46, 22, 3, 14, 7, 21, 24, 20, 24, 23, 45, 31, 50, 34, 17, 6, 29, 27, 17,
    6, 23, 10, 5, 18, 23, 10, 21, 6, 37, 25, 23, 9, 61, 21, 41, 9, 22, 10, 17,
    33, 20, 33, 15, 15, 36, 25, 27, 33, 24, 9, 19, 36, 19, 30, 14, 49, 14, 21, 15,
    51, 31, 10, 24, 23, 22, 26, 67, 52, 30, 16, 13, 17, 27, 30, 40, 3, 27, 14, 13,
    13, 18, 17, 18, 12, 13, 15, 39, 10, 16, 45, 47, 10, 34, 13, 28, 21, 30, 42, 34,
    85, 15, 24, 46, 116, 33, 22, 36, 35, 11, 4, 12, 16, 21, 64, 15, 14, 5, 35, 39,
    19, 22, 18, 12, 28, 14, 27, 20, 78, 12, 20, 50, 28, 17, 28, 7, 26, 19, 29, 21,
    59, 9, 164, 55, 23, 19, 42, 55, 20, 14, 59, 32, 15, 27, 25, 6, 16, 11, 21, 13,
    13, 46, 11, 6, 52, 28, 16, 15, 4, 35, 33, 21, 15, 15, 14, 45, 19, 10, 47, 79,
    26, 40, 33, 44, 22, 17, 19, 13, 24, 8, 19, 18, 32, 65, 34, 27, 36, 25, 65, 24,
    14, 23, 69, 31, 29, 22, 24, 63, 17, 21, 25, 30, 29, 16, 13, 42, 24, 30, 26, 24,
    45, 52, 18, 25, 50, 23, 38, 36, 22, 19, 33, 16, 22, 10, 15, 6, 8, 12, 33, 19,
    55, 11, 13, 12, 20, 10, 7, 28, 35, 38, 25, 41, 16, 42, 4, 29, 17, 39, 29, 15,
    12, 10, 15, 17, 23, 14, 20, 6, 10, 23, 31, 36, 34, 14, 73, 15, 10, 18, 11, 10,
    14, 31, 19, 25, 40, 95, 9, 46, 24, 29, 48, 41, 8, 10, 29, 30, 10, 23, 15, 12,
    24, 13, 15, 49, 42, 34, 16, 16, 38, 18, 29, 27, 31, 51, 29, 25, 15, 22, 50, 19,
    23, 42, 36, 12, 8, 27, 16, 13, 12, 54, 57, 28, 17, 23, 39, 37, 56, 19, 24, 28,
    14, 27, 14, 22, 33, 35, 11, 40, 19, 26, 20, 33, 19, 16, 15, 20, 30, 10, 81, 27,
    41, 15, 27, 11, 20, 18, 17, 51, 18, 43, 19, 17, 19, 19, 28, 43, 29, 24, 43, 25,
    28, 21, 32, 10, 67, 20, 40, 28, 14, 34, 30, 20, 14, 32, 15, 37, 36, 30, 67, 20,
    16, 63, 8, 71, 9, 15, 32, 9, 15, 20, 55, 22, 42, 25, 22, 16, 16, 42, 10, 75,
    40, 13, 24, 31, 27, 17, 12, 15, 40, 15, 36, 46, 33, 19, 28, 15, 22, 10, 51, 30,
    48, 42, 22, 58, 20, 22, 19, 22, 12, 24, 13, 39, 27, 28, 16, 17, 12, 14, 22, 11,
    17, 22, 26, 9, 41, 12, 57, 15, 23, 35, 45, 14, 21, 10, 14, 8, 14, 26, 16, 27,
    12, 21, 68, 49, 7, 18, 23, 48, 16, 16, 64, 25, 24, 13, 36, 14, 42, 18, 37, 13,
    19, 20, 55, 16, 11, 13, 29, 24, 22, 16, 15, 14, 25, 29, 30, 26, 7, 17, 12, 19,
    22, 32, 26, 38, 27, 6, 31, 80, 39, 100, 38, 8, 20, 22, 112, 24, 16, 24, 10, 77,
    12, 19, 26, 16, 24, 10, 35, 8, 13, 28, 88, 21, 10, 27, 15, 27, 26, 47, 69, 39,
    28, 13, 23, 22, 19, 21, 17, 45, 41, 26, 31, 15, 11, 71, 27, 18, 8, 14, 19, 13,
    19, 21, 15, 22, 10, 21, 16, 17, 5, 42, 56, 17, 11, 8, 19, 17, 20, 19, 16, 7,
    45, 32, 14, 18, 18, 13, 17, 20, 26, 22, 21, 15, 25, 16, 17, 53, 11, 53, 28, 25,
    27, 13, 34, 19, 30, 14, 18, 24, 93, 10, 31, 19, 20, 21, 45, 30, 68, 71, 26, 16,
    14, 19, 9, 18, 43, 24, 23, 51, 36, 29, 44, 10, 17, 10, 18, 22, 9, 11, 21, 37,
    55, 25, 24, 33, 63, 17, 48, 8, 30, 46, 63, 27, 37, 30, 41, 18, 13, 13, 8, 18,
    16, 53, 10, 9, 44, 38, 25, 49, 30, 15, 37, 7, 26, 77, 14, 14, 28, 47, 34, 21,
    15, 43, 9, 29, 35, 19, 22, 38, 32, 29, 21, 19, 40, 22, 63, 24, 20, 10, 9, 5,
    14, 26, 16, 18, 136, 11, 29, 34, 23, 22, 34, 25, 20, 40, 10, 9, 20, 11, 36, 21,
    47, 33, 31, 23, 10, 15, 59, 5, 13, 21, 59, 77, 26, 47, 16, 10, 108, 15, 8, 22,
    21, 16, 73, 22, 21, 12, 7, 13, 41, 6, 6, 11, 16, 18, 16, 47, 57, 22, 43, 21,
    16, 30, 25, 14, 36, 31, 23, 10, 32, 8, 23, 15, 40, 35, 12, 16, 42, 71, 39, 15,
    9, 11, 78, 60, 55, 51, 22, 62, 25, 35, 34, 20, 25, 20, 15, 46, 12, 18, 50, 26,
    20, 27, 131, 9, 37, 69, 40, 30, 50, 13, 36, 55, 51, 14, 22, 32, 43, 42, 13, 11,
    11, 62, 39, 24, 14, 14, 29, 21, 9, 24, 11, 17, 14, 15, 22, 43, 13, 29, 23, 33,
    21, 33, 17, 32, 24, 30, 49, 27, 34, 16, 15, 69, 19, 7, 14, 30, 52, 16, 27, 25,
    15, 26, 24, 12, 16, 5, 23, 17, 8, 25, 15, 18, 20, 46, 17, 44, 108, 47, 69, 24,
    18, 7, 24, 36, 49, 15, 24, 8, 13, 27, 14, 54, 9, 52, 97, 14, 49, 21, 11, 8,
    79, 21, 11, 30, 14, 14, 14, 9, 21, 9, 62, 23, 12, 36, 25, 19, 12, 27, 35, 19,
    27, 13, 21, 24, 70, 20, 35, 18, 48, 8, 14, 28, 53, 12, 14, 19, 19, 15, 16, 33,
    17, 27, 78, 101, 13, 40, 18, 20, 39, 9, 83, 29, 20, 10, 42, 26, 8, 75, 27, 17,
    43, 7, 12, 20, 29, 10, 16, 51, 19, 24, 45, 18, 15, 26, 13, 15, 21, 38, 71, 13,
    18, 44, 13, 39, 30, 53, 31, 12, 34, 55, 19, 84, 12, 24, 29, 22, 59, 22, 47, 17,
    11, 36, 50, 53, 51, 28, 22, 26, 6, 19, 14, 47, 30, 15, 16, 14, 12, 19, 35, 24,
    16, 22, 13, 13, 25, 16, 13, 16, 59, 24, 30, 22, 17, 6, 33, 19, 47, 22, 29, 22,
    11, 32, 21, 23, 19, 74, 17, 15, 44, 34, 52, 11, 36, 14, 21, 10, 9, 42, 34, 28,
    78, 26, 17, 36, 26, 14, 10, 8, 81, 19, 28, 13, 25, 38, 38, 19, 24, 11, 14, 17,
    13, 52, 14, 44, 43, 58, 29, 8, 34, 53, 17, 24, 39, 32, 13, 39, 36, 22, 54, 18,
    35, 23, 31, 64, 27, 22, 10, 16, 14, 31, 21, 38, 31, 5, 13, 21, 42, 13, 25, 66,
    11, 20, 42, 30, 42, 27, 30, 41, 35, 17, 19, 24, 38, 34, 33, 33, 16, 27, 46, 20,
    77, 19, 92, 66, 23, 8, 13, 56, 17, 11, 19, 70, 16, 13, 17, 24, 27, 14, 16, 45,
    45, 14, 14, 50, 14, 5, 26, 14, 21, 10, 48, 6, 9, 17, 27, 17, 94, 11, 28, 49,
    10, 16, 7, 45, 9, 25, 28, 19, 12, 15, 43, 29, 22, 24, 26, 39, 15, 39, 27, 30,
    17, 33, 24, 72, 9, 19, 10, 19, 19, 11, 43, 9, 27, 15, 14, 19, 42, 44, 57, 17,
    57, 31, 22, 44, 33, 21, 16, 47, 11, 28, 16, 23, 107, 41, 37, 31, 25, 20, 19, 58,
    16, 8, 12, 36, 7, 21, 28, 10, 12, 13, 103, 42, 13, 11, 26, 21, 25, 12, 14, 26,
    25, 12, 25, 38, 32, 16, 35, 15, 14, 12, 35, 31, 8, 20, 17, 15, 19, 10, 12, 83,
    17, 8, 38, 10, 42, 16, 27, 24, 38, 88, 24, 20, 14, 30, 9, 21, 30, 49, 13, 19,
    28, 37, 29, 31, 20, 18, 46, 18, 27, 28, 34, 55, 18, 11, 21, 69, 39, 18, 14, 5,
    7, 21, 16, 33, 19, 15, 22, 14, 32, 59, 16, 41, 9, 23, 9, 17, 44, 24, 41, 9,
    18, 42, 10, 26, 19, 11, 14, 8, 27, 8, 60, 28, 24, 39, 44, 27, 42, 14, 12, 35,
    23, 9, 13, 43, 26, 16, 22, 50, 60, 37, 17, 28, 27, 28, 31, 25, 26, 21, 61, 34,
    20, 13, 61, 7, 28, 14, 22, 34, 21, 16, 18, 29, 9, 23, 5, 28, 8, 11, 25, 32,
    28, 33, 18, 77, 13, 14, 42, 26, 43, 19, 23, 32, 33, 18, 10, 24, 54, 28, 50, 12,
    59, 43, 85, 10, 40, 44, 30, 29, 23, 20, 22, 31, 12, 13, 13, 8, 23, 49, 38, 34,
    18, 23, 11, 38, 22, 17, 19, 58, 15, 14, 50, 26, 7, 9, 20, 18, 16, 18, 25, 13,
    14, 39, 20, 77, 34, 20, 34, 27, 16, 37, 16, 25, 32, 13, 15, 75, 44, 9, 24, 17,
    15, 7, 11, 36, 12, 54, 38, 53, 16, 12, 13, 18, 34, 21, 34, 24, 22, 79, 31, 22,
    13, 16, 10, 25, 8, 16, 15, 21, 34, 13, 13, 44, 13, 29, 45, 44, 53, 9, 32, 31,
    28, 20, 16, 20, 11, 35, 19, 12, 122, 15, 34, 21, 31, 26, 21, 9, 29, 27, 27, 28,
    10, 17, 22, 47, 27, 88, 13, 32, 49, 18, 21, 64, 18, 11, 31, 17, 9, 31, 7, 31,
    7, 40, 8, 46, 9, 50, 38, 23, 62, 12, 19, 26, 61, 32, 17, 35, 88, 29, 26, 22,
    27, 32, 18, 29, 10, 21, 23, 15, 28, 34, 40, 51, 22, 27, 18, 15, 40, 44, 23, 25,
    8, 22, 21, 18, 22, 15, 7, 18, 28, 58, 32, 35, 26, 51, 12, 34, 10, 17, 11, 27,
    16, 18, 45, 27, 30, 27, 13, 36, 21, 19, 30, 29, 28, 15, 17, 19, 25, 26, 14, 15,
    6, 30, 30, 25, 60, 18, 66, 34, 15, 23, 13, 25, 33, 47, 18, 17, 17, 36, 32, 36,
    35, 29, 62, 15, 10, 11, 52, 23, 32, 23, 25, 25, 26, 10, 39, 23, 16, 9, 35, 42,
    52, 17, 7, 13, 58, 16, 29, 45, 20, 27, 21, 14, 42, 18, 40, 39, 20, 18, 23, 24,
    18, 24, 4, 24, 18, 26, 44, 12, 25, 8, 7, 16, 38, 10, 18, 16, 26, 13, 29, 12,
    61, 22, 29, 28, 23, 17, 12, 31, 37, 24, 22, 14, 18, 21, 55, 18, 15, 4, 17, 12,
    14, 49, 24, 20, 15, 21, 7, 21, 63, 6, 28, 21, 20, 29, 24, 11, 32, 44, 22, 33,
    17, 36, 17, 34, 7, 33, 24, 24, 22, 21, 26, 12, 7, 54, 31, 63, 34, 21, 25, 13,
    9, 20, 39, 45, 43, 24, 38, 20, 38, 29, 22, 7, 27, 12, 18, 45, 52, 12, 20, 22,
    21, 6, 52, 43, 7, 19, 31, 15, 27, 20, 12, 11, 16, 22, 27, 19, 42, 53, 29, 48,
    17, 29, 13, 43, 12, 21, 14, 16, 48, 28, 27, 17, 14, 38, 40, 55, 38, 11, 16, 27,
    23, 16, 84, 22, 14, 47, 8, 18, 36, 17, 26, 15, 12, 31, 21, 21, 49, 59, 25, 10,
    15, 35, 21, 14, 13, 36, 31, 10, 31, 19, 18, 47, 19, 8, 21, 5, 11, 20, 19, 23,
    72, 24, 10, 24, 87, 17, 19, 15, 22, 32, 11, 37, 24, 9, 25, 13, 38, 52, 12, 37,
    29, 11, 34, 11, 21, 25, 21, 86, 28, 4, 31, 33, 23, 32, 34, 38, 54, 19, 19, 17,
    25, 30, 53, 27, 22, 39, 11, 43, 38, 69, 14, 11, 65, 8, 11, 12, 35, 20, 26, 46,
    17, 17, 96, 33, 11, 30, 24, 39, 10, 28, 9, 14, 52, 22, 19, 49, 22, 22, 37, 21,
    42, 12, 19, 126, 15, 30, 35, 11, 29, 34, 23, 70, 28, 74, 23, 21, 30, 8, 17, 19,
    34, 14, 33, 8, 30, 49, 22, 8, 15, 44, 12, 14, 40, 21, 18, 105, 28, 9, 19, 16,
    24, 16, 22, 40, 18, 22, 46, 16, 15, 42, 57, 37, 49, 28, 19, 28, 28, 58, 24, 42,
    18, 9, 10, 11, 15, 31, 8, 25, 28, 22, 15, 31, 16, 34, 12, 51, 28, 29, 24, 11,
    36, 12, 36, 16, 45, 18, 25, 20, 20, 30, 107, 14, 20, 15, 19, 6, 15, 19, 10, 23,
    20, 9, 34, 29, 8, 21, 127, 20, 24, 26, 47, 40, 24, 26, 15, 80, 30, 8, 23, 22,
    21, 13, 27, 26, 25, 39, 19, 16, 33, 47, 102, 57, 40, 21, 21, 35, 24, 34, 22, 11,
    31, 25, 32, 66, 18, 14, 49, 30, 37, 22, 31, 24, 22, 34, 28, 15, 21, 22, 17, 41,
    19, 21, 28, 13, 37, 21, 9, 6, 30, 17, 34, 18, 40, 20, 40, 13, 39, 20, 62, 27,
    41, 15, 29, 16, 6, 27, 9, 18, 54, 38, 27, 65, 31, 13, 15, 30, 9, 23, 32, 32,
    10, 39, 51, 30, 32, 86, 41, 13, 26, 7, 24, 13, 19, 9, 13, 40, 47, 48, 14, 39,
    13, 15, 16, 65, 41, 37, 21, 38, 39, 9, 41, 83, 7, 4, 23, 12, 14, 7, 89, 40,
    23, 14, 34, 30, 28, 22, 65, 21, 13, 88, 36, 31, 37, 18, 16, 10, 50, 47, 23, 7,
    7, 13, 22, 12, 33, 36, 45, 25, 43, 23, 26, 11, 40, 33, 41, 16, 25, 6, 23, 18,
    29, 9, 222, 33, 57, 12, 35, 24, 27, 27, 27, 67, 40, 18, 14, 18, 14, 20, 34, 54,
    34, 48, 12, 13, 19, 19, 55, 75, 34, 25, 21, 15, 14, 11, 21, 24, 14, 21, 77, 37,
    21, 19, 29, 29, 27, 13, 22, 32, 38, 15, 14, 19, 10, 29, 18, 38, 153, 7, 31, 20,
    22, 75, 11, 26, 31, 12, 9, 8, 17, 15, 33, 39, 15, 26, 9, 54, 17, 32, 47, 14,
    68, 11, 73, 22, 13, 7, 11, 23, 31, 75, 34, 10, 34, 7, 23, 29, 11, 33, 22, 50,
    15, 27, 16, 50, 35, 17, 18, 10, 13, 17, 44, 35, 30, 21, 24, 15, 13, 13, 57, 16,
    26, 8, 8, 13, 30, 17, 49, 54, 50, 38, 9, 25, 21, 15, 49, 18, 43, 29, 15, 64,
    37, 16, 11, 37, 9, 25, 18, 17, 57, 14, 26, 25, 11, 22, 16, 34, 15, 28, 31, 7,
    22, 47, 27, 18, 23, 11, 36, 39, 21, 48, 21, 74, 49, 22, 10, 23, 14, 31, 27, 13,
    25, 12, 42, 8, 43, 41, 18, 46, 59, 37, 105, 21, 44, 18, 20, 12, 24, 15, 13, 72,
    15, 83, 12, 25, 20, 28, 13, 112, 18, 20, 10, 30, 7, 7, 8, 14, 58, 39, 67, 17,
    8, 22, 22, 17, 12, 47, 17, 30, 16, 23, 30, 40, 9, 16, 19, 16, 22, 68, 13, 24,
    15, 22, 21, 6, 16, 6, 12, 32, 19, 34, 15, 32, 18, 11, 17, 79, 29, 26, 39, 59,
    12, 13, 20, 17, 45, 15, 24, 6, 43, 18, 34, 9, 28, 7, 9, 11, 18, 24, 5, 39,
    65, 19, 22, 17, 37, 15, 26, 61, 27, 10, 39, 11, 27, 42, 24, 43, 47, 30, 20, 22,
    17, 20, 16, 17, 21, 9, 10, 24, 35, 16, 10, 12, 9, 25, 31, 18, 38, 18, 17, 61,
    17, 48, 32, 60, 18, 12, 30, 20, 6, 11, 15, 34, 19, 4, 7, 52, 4, 34, 45, 8,
    13, 21, 22, 30, 24, 11, 17, 30, 57, 55, 34, 17, 9, 16, 44, 17, 17, 16, 9, 20,
    10, 26, 21, 11, 39, 13, 18, 25, 22, 15, 34, 6, 47, 54, 45, 23, 19, 22, 21, 18,
    36, 49, 27, 18, 31, 22, 19, 76, 37, 31, 16, 7, 36, 38, 23, 15, 35, 19, 60, 9,
    24, 39, 14, 19, 23, 28, 27, 45, 12, 8, 18, 15, 13, 104, 35, 11, 21, 29, 45, 51,
    32, 19, 24, 12, 30, 11, 13, 12, 61, 26, 34, 52, 14, 15, 31, 49, 24, 15, 21, 17,
    12, 59, 9, 48, 49, 6, 32, 23, 21, 18, 17, 13, 22, 22, 17, 20, 28, 18, 27, 25,
    20, 24, 15, 13, 70, 17, 26, 44, 23, 45, 16, 15, 14, 22, 32, 28, 42, 37, 20, 4,
    22, 13, 25, 26, 3, 15, 46, 19, 22, 28, 38, 50, 56, 28, 9, 21, 62, 44, 74, 29,
    25, 30, 19, 56, 24, 17, 20, 13, 22, 9, 16, 23, 12, 27, 28, 28, 46, 24, 22, 54,
    21, 25, 18, 32, 44, 18, 90, 58, 64, 26, 17, 17, 38, 44, 18, 10, 25, 17, 13, 24,
    16, 19, 25, 33, 51, 25, 42, 16, 23, 15, 52, 76, 26, 9, 66, 15, 19, 17, 22, 24,
    7, 45, 14, 30, 45, 7, 17, 41, 28, 30, 20, 11, 22, 24, 18, 20, 40, 7, 8, 56,
    32, 40, 16, 31, 43, 51, 24, 28, 14, 9, 15, 19, 62, 82, 34, 49, 21, 29, 24, 16,
    30, 19, 48, 13, 16, 66, 33, 36, 14, 37, 31, 23, 27, 55, 20, 71, 26, 48, 27, 5,
    43, 25, 11, 7, 24, 13, 29, 8, 32, 24, 23, 14, 18, 26, 35, 43, 15, 61, 16, 15,
    43, 5, 31, 9, 13, 24, 53, 14, 11, 25, 9, 20, 20, 21, 44, 22, 62, 22, 15, 22,
    29, 28, 48, 15, 21, 15, 48, 32, 20, 18, 9, 6, 71, 30, 23, 22, 20, 52, 45, 14,
    5, 41, 21, 12, 17, 19, 18, 18, 70, 12, 14, 26, 27, 20, 42, 29, 9, 12, 23, 39,
    18, 15, 4, 18, 18, 30, 14, 17, 43, 60, 16, 33, 13, 20, 17, 14, 8, 15, 37, 31,
    50, 18, 8, 12, 19, 26, 42, 13, 11, 29, 36, 55, 21, 7, 28, 11, 22, 27, 21, 106,
    15, 17, 53, 11, 12, 21, 32, 15, 25, 21, 18, 6, 17, 21, 10, 28, 28, 19, 36, 16,
    23, 15, 36, 21, 32, 8, 15, 28, 52, 17, 68, 16, 8, 11, 58, 24, 23, 108, 32, 14,
    14, 8, 7, 22, 24, 8, 45, 19, 43, 25, 16, 43, 17, 72, 10, 18, 43, 14, 20, 41,
    23, 19, 16, 39, 14, 21, 23, 16, 10, 23, 20, 17, 32, 24, 17, 15, 23, 16, 95, 23,
    19, 41, 15, 18, 53, 54, 49, 34, 47, 34, 13, 38, 55, 31, 35, 19, 23, 32, 26, 22,
    27, 27, 23, 11, 9, 23, 21, 11, 22, 39, 52, 13, 18, 25, 27, 47, 25, 27, 5, 20,
    10, 42, 19, 23, 22, 21, 26, 11, 57, 12, 10, 7, 43, 31, 11, 95, 18, 17, 28, 33,
]
print(f"Ready: {len(population)} tickets, true mean {statistics.mean(population):.1f} minutes")

## Setup: the rated tickets

The next cell defines `rated`: the reply times of the 2,155 tickets of the population that a customer rated. Customers rate more often when the reply came quickly.

Run the cell. You should see `Ready: 2155 rated tickets, mean 20.9 minutes`.

In [ ]:
# The data, written in full (you do not need to read it).
rated = [
    24, 11, 32, 15, 20, 11, 14, 15, 20, 24, 22, 34, 10, 24, 19, 18, 14, 17, 14, 14,
    15, 90, 15, 23, 39, 24, 24, 13, 29, 12, 18, 10, 27, 30, 32, 13, 9, 18, 10, 11,
    16, 7, 26, 24, 15, 20, 13, 11, 6, 12, 24, 10, 13, 23, 19, 10, 19, 16, 30, 18,
    13, 13, 16, 19, 33, 15, 29, 17, 13, 25, 18, 9, 8, 25, 22, 14, 30, 24, 31, 21,
    35, 8, 22, 10, 10, 30, 17, 9, 14, 12, 28, 64, 27, 9, 10, 11, 11, 18, 10, 15,
    30, 47, 15, 27, 78, 20, 12, 29, 19, 51, 13, 13, 18, 31, 32, 14, 29, 40, 16, 37,
    38, 12, 13, 18, 13, 10, 10, 32, 24, 29, 22, 13, 31, 23, 15, 18, 33, 22, 20, 21,
    11, 25, 19, 9, 18, 15, 56, 7, 26, 24, 7, 15, 21, 19, 53, 18, 20, 10, 27, 16,
    11, 10, 37, 17, 29, 45, 20, 18, 26, 28, 30, 16, 17, 8, 33, 28, 23, 13, 30, 25,
    10, 5, 31, 30, 10, 12, 20, 13, 29, 20, 14, 40, 15, 51, 17, 26, 9, 17, 13, 37,
    17, 106, 5, 25, 31, 16, 20, 7, 24, 23, 9, 19, 25, 32, 11, 12, 30, 16, 14, 21,
    18, 11, 15, 54, 12, 16, 18, 25, 59, 13, 24, 18, 28, 9, 14, 63, 10, 17, 20, 19,
    19, 9, 5, 26, 23, 42, 34, 15, 21, 17, 12, 8, 20, 17, 22, 30, 17, 12, 7, 48,
    28, 14, 15, 40, 38, 13, 26, 3, 26, 18, 34, 16, 26, 9, 7, 17, 10, 16, 7, 20,
    4, 34, 19, 33, 8, 33, 59, 10, 12, 13, 12, 8, 12, 29, 9, 37, 19, 20, 27, 16,
    33, 10, 31, 20, 22, 22, 7, 33, 11, 14, 8, 40, 39, 20, 37, 18, 18, 17, 9, 58,
    13, 14, 8, 25, 12, 19, 20, 45, 9, 33, 37, 28, 7, 24, 21, 11, 111, 16, 16, 38,
    3, 6, 7, 8, 17, 39, 12, 36, 11, 9, 12, 22, 51, 18, 14, 12, 11, 17, 18, 24,
    16, 15, 32, 15, 17, 6, 30, 13, 8, 7, 49, 6, 24, 16, 16, 111, 6, 20, 11, 9,
    18, 16, 20, 11, 17, 13, 10, 10, 16, 19, 29, 6, 7, 53, 17, 29, 36, 15, 16, 46,
    11, 5, 6, 17, 16, 16, 17, 3, 19, 12, 19, 7, 12, 16, 34, 93, 14, 18, 20, 10,
    18, 13, 18, 6, 29, 15, 30, 8, 15, 16, 8, 9, 19, 22, 20, 24, 27, 16, 21, 19,
    17, 9, 22, 18, 55, 12, 9, 18, 15, 8, 8, 14, 15, 29, 44, 11, 13, 16, 25, 30,
    7, 31, 21, 23, 13, 25, 6, 9, 16, 20, 11, 58, 12, 34, 28, 17, 12, 14, 18, 9,
    13, 18, 12, 22, 11, 22, 15, 12, 15, 20, 31, 18, 17, 26, 21, 28, 12, 20, 18, 24,
    9, 20, 42, 65, 23, 38, 19, 10, 15, 14, 12, 19, 25, 15, 6, 15, 16, 11, 18, 17,
    21, 35, 23, 15, 65, 38, 20, 17, 11, 10, 11, 16, 15, 29, 10, 36, 40, 25, 34, 6,
    32, 17, 9, 72, 34, 20, 9, 15, 16, 12, 14, 24, 21, 10, 30, 29, 16, 23, 28, 6,
    29, 8, 7, 15, 15, 29, 27, 48, 23, 36, 41, 23, 29, 33, 50, 19, 43, 13, 6, 21,
    18, 20, 11, 30, 44, 15, 21, 26, 15, 8, 19, 14, 38, 38, 13, 20, 32, 16, 22, 17,
    6, 17, 22, 24, 14, 28, 28, 32, 17, 33, 34, 16, 20, 37, 29, 40, 27, 32, 9, 32,
    13, 16, 13, 20, 22, 19, 22, 30, 41, 26, 19, 14, 16, 11, 14, 63, 28, 16, 30, 37,
    9, 27, 15, 22, 18, 35, 20, 23, 20, 13, 13, 19, 14, 17, 9, 16, 23, 18, 36, 17,
    16, 32, 7, 15, 20, 15, 15, 36, 21, 13, 32, 17, 13, 17, 17, 9, 24, 17, 33, 40,
    12, 13, 17, 40, 20, 35, 20, 20, 3, 16, 17, 34, 6, 17, 31, 31, 9, 18, 11, 20,
    39, 8, 11, 20, 27, 28, 18, 64, 19, 40, 26, 22, 17, 15, 18, 13, 35, 15, 9, 15,
    24, 18, 16, 23, 13, 12, 11, 20, 22, 23, 29, 15, 19, 11, 12, 7, 32, 17, 18, 19,
    31, 11, 10, 6, 11, 13, 25, 9, 22, 18, 16, 25, 20, 30, 18, 16, 13, 8, 16, 12,
    30, 33, 31, 43, 16, 21, 72, 24, 10, 9, 7, 14, 41, 8, 14, 19, 36, 19, 29, 13,
    29, 16, 11, 24, 13, 11, 23, 28, 30, 21, 22, 29, 16, 17, 19, 10, 34, 16, 9, 27,
    9, 20, 11, 12, 18, 24, 17, 16, 35, 36, 17, 23, 15, 30, 38, 29, 19, 17, 10, 23,
    13, 10, 17, 42, 17, 11, 6, 40, 20, 48, 15, 13, 32, 12, 40, 34, 18, 23, 13, 35,
    14, 10, 17, 21, 20, 19, 9, 28, 19, 14, 21, 8, 27, 21, 12, 24, 12, 28, 20, 28,
    40, 23, 19, 10, 22, 12, 11, 30, 44, 12, 31, 40, 18, 12, 17, 25, 34, 53, 14, 61,
    8, 19, 21, 19, 4, 26, 22, 14, 47, 24, 9, 22, 16, 16, 17, 11, 14, 11, 28, 20,
    19, 9, 17, 17, 21, 24, 24, 14, 12, 8, 25, 14, 38, 7, 22, 9, 10, 38, 8, 15,
    28, 8, 19, 15, 12, 18, 20, 13, 18, 17, 24, 8, 12, 24, 17, 12, 11, 16, 13, 8,
    16, 7, 16, 27, 6, 22, 19, 29, 39, 16, 34, 12, 10, 13, 17, 22, 18, 26, 13, 31,
    14, 5, 12, 17, 20, 14, 19, 38, 15, 14, 19, 15, 26, 17, 8, 10, 17, 12, 38, 18,
    19, 8, 71, 22, 11, 8, 12, 8, 24, 14, 14, 31, 5, 11, 6, 19, 21, 18, 11, 17,
    14, 37, 18, 18, 18, 20, 13, 15, 39, 14, 17, 38, 10, 29, 21, 14, 16, 6, 16, 9,
    46, 31, 29, 3, 7, 21, 20, 31, 27, 6, 5, 23, 10, 6, 37, 9, 9, 22, 10, 33,
    15, 15, 36, 33, 19, 14, 10, 13, 40, 14, 12, 39, 13, 28, 21, 30, 42, 34, 116, 22,
    35, 14, 19, 18, 12, 20, 12, 28, 7, 19, 164, 55, 23, 19, 25, 6, 16, 21, 13, 13,
    11, 6, 21, 15, 14, 79, 17, 13, 8, 19, 32, 65, 24, 14, 25, 38, 19, 10, 15, 6,
    33, 11, 13, 12, 20, 7, 28, 35, 41, 4, 17, 15, 12, 14, 20, 6, 10, 34, 18, 11,
    10, 31, 29, 10, 15, 12, 16, 16, 29, 22, 23, 12, 17, 24, 14, 20, 16, 20, 10, 15,
    27, 11, 20, 18, 17, 18, 19, 19, 19, 28, 29, 32, 28, 14, 30, 20, 8, 9, 10, 40,
    31, 27, 17, 15, 40, 15, 15, 20, 12, 16, 41, 15, 23, 21, 10, 14, 14, 26, 16, 27,
    12, 16, 16, 64, 25, 13, 14, 18, 20, 15, 17, 12, 27, 6, 8, 22, 12, 35, 28, 10,
    15, 39, 28, 13, 23, 19, 17, 26, 31, 15, 8, 14, 19, 21, 16, 17, 5, 56, 17, 8,
    20, 16, 18, 17, 26, 16, 17, 11, 13, 19, 30, 18, 10, 31, 30, 71, 14, 43, 23, 36,
    44, 22, 11, 25, 8, 30, 27, 18, 13, 13, 8, 9, 38, 15, 7, 26, 14, 14, 28, 21,
    15, 9, 21, 19, 9, 14, 18, 29, 23, 34, 10, 9, 20, 11, 31, 15, 13, 77, 16, 8,
    21, 16, 73, 12, 13, 6, 6, 18, 16, 21, 30, 25, 14, 31, 32, 23, 15, 12, 39, 15,
    11, 25, 20, 15, 18, 26, 9, 13, 11, 24, 14, 14, 29, 21, 9, 24, 17, 15, 23, 33,
    33, 17, 27, 34, 16, 15, 7, 14, 52, 16, 27, 25, 16, 8, 15, 18, 7, 24, 15, 8,
    13, 14, 11, 14, 9, 21, 19, 12, 35, 27, 21, 70, 35, 8, 14, 28, 19, 15, 16, 17,
    40, 20, 20, 10, 17, 7, 12, 29, 16, 13, 15, 13, 30, 19, 84, 24, 29, 17, 11, 53,
    22, 19, 14, 15, 16, 19, 24, 22, 25, 30, 6, 33, 29, 22, 21, 11, 36, 10, 14, 10,
    8, 19, 13, 38, 19, 14, 14, 34, 24, 32, 39, 22, 35, 23, 22, 10, 14, 31, 21, 13,
    13, 11, 30, 35, 17, 24, 38, 34, 20, 19, 23, 8, 13, 56, 17, 11, 16, 13, 17, 24,
    14, 14, 50, 5, 10, 48, 17, 27, 10, 7, 9, 28, 19, 15, 22, 26, 15, 39, 27, 33,
    10, 19, 15, 19, 57, 17, 44, 21, 11, 31, 25, 20, 16, 10, 13, 42, 11, 21, 25, 14,
    26, 25, 12, 38, 35, 14, 12, 20, 19, 10, 8, 10, 16, 27, 24, 20, 14, 9, 30, 19,
    29, 31, 34, 18, 11, 14, 7, 19, 32, 59, 16, 41, 9, 41, 9, 19, 11, 14, 28, 39,
    14, 9, 13, 50, 17, 31, 7, 22, 16, 29, 23, 5, 8, 11, 25, 33, 18, 14, 23, 32,
    33, 24, 50, 12, 59, 44, 30, 29, 20, 31, 12, 13, 8, 23, 38, 18, 17, 15, 16, 18,
    13, 14, 39, 16, 37, 13, 15, 24, 17, 36, 12, 16, 12, 13, 18, 34, 31, 22, 16, 10,
    16, 15, 13, 9, 31, 20, 16, 12, 15, 31, 21, 28, 18, 11, 7, 9, 23, 19, 32, 17,
    26, 32, 18, 29, 10, 28, 34, 22, 40, 44, 25, 8, 22, 18, 22, 32, 12, 10, 11, 27,
    27, 13, 21, 28, 17, 19, 14, 6, 18, 18, 17, 36, 29, 11, 52, 32, 23, 10, 39, 42,
    17, 13, 29, 20, 27, 21, 18, 24, 24, 18, 26, 8, 7, 16, 38, 13, 29, 28, 17, 12,
    22, 18, 21, 29, 17, 17, 7, 33, 24, 21, 26, 12, 34, 21, 13, 9, 20, 39, 20, 38,
    29, 27, 12, 7, 27, 22, 27, 19, 13, 12, 17, 14, 38, 40, 11, 16, 14, 18, 26, 15,
    12, 21, 25, 21, 14, 13, 31, 19, 8, 19, 23, 10, 17, 19, 9, 12, 37, 29, 11, 11,
    21, 19, 19, 25, 11, 11, 20, 26, 17, 11, 10, 9, 14, 22, 22, 21, 12, 15, 11, 34,
    19, 14, 15, 18, 28, 9, 19, 16, 22, 18, 16, 15, 28, 19, 28, 9, 10, 11, 25, 28,
    16, 28, 24, 36, 16, 18, 14, 15, 19, 10, 20, 34, 29, 40, 8, 22, 13, 39, 19, 35,
    24, 22, 25, 32, 66, 18, 30, 37, 31, 24, 17, 41, 19, 13, 37, 9, 6, 30, 17, 34,
    18, 20, 13, 20, 27, 15, 6, 18, 38, 13, 15, 32, 10, 7, 13, 9, 48, 14, 15, 16,
    37, 7, 4, 12, 14, 22, 13, 36, 16, 7, 13, 33, 18, 35, 24, 27, 40, 18, 14, 20,
    13, 19, 34, 21, 15, 21, 24, 21, 22, 32, 38, 15, 14, 10, 11, 17, 15, 39, 26, 54,
    11, 73, 13, 34, 10, 7, 23, 11, 16, 18, 13, 30, 24, 15, 13, 16, 8, 8, 13, 30,
    17, 38, 9, 15, 18, 16, 11, 17, 14, 16, 15, 18, 11, 21, 22, 14, 31, 27, 13, 12,
    42, 18, 46, 18, 20, 12, 24, 13, 12, 20, 13, 20, 10, 7, 8, 22, 12, 16, 30, 9,
    16, 19, 13, 6, 16, 6, 12, 34, 32, 18, 11, 26, 59, 17, 6, 18, 34, 9, 7, 18,
    5, 17, 15, 10, 24, 47, 20, 16, 17, 21, 9, 16, 12, 9, 25, 18, 48, 30, 6, 15,
    19, 4, 4, 34, 8, 13, 21, 9, 17, 16, 26, 21, 11, 39, 18, 25, 15, 34, 19, 18,
    22, 19, 37, 31, 38, 9, 14, 28, 18, 35, 30, 61, 14, 9, 6, 23, 18, 13, 17, 20,
    20, 15, 13, 45, 15, 28, 20, 4, 15, 19, 22, 28, 29, 56, 16, 12, 24, 22, 25, 18,
    18, 17, 18, 10, 13, 16, 25, 25, 15, 9, 15, 19, 7, 7, 17, 20, 22, 18, 20, 40,
    8, 56, 16, 9, 19, 24, 30, 48, 16, 36, 37, 31, 23, 20, 27, 43, 25, 7, 24, 32,
    35, 13, 14, 11, 9, 20, 20, 22, 22, 29, 15, 15, 18, 9, 20, 14, 12, 70, 14, 20,
    42, 9, 12, 23, 18, 4, 18, 30, 17, 33, 20, 14, 8, 12, 26, 13, 29, 36, 28, 22,
    106, 15, 17, 11, 12, 25, 21, 6, 17, 10, 28, 19, 15, 36, 21, 32, 8, 8, 24, 22,
    8, 19, 25, 16, 17, 10, 20, 23, 39, 14, 10, 20, 17, 17, 23, 95, 19, 15, 34, 13,
    31, 19, 32, 27, 23, 9, 23, 19, 11, 10, 7, 31, 95, 18, 33,
]
print(f"Ready: {len(rated)} rated tickets, mean {statistics.mean(rated):.1f} minutes")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Population and sample

The **population** is the whole group that you want to know about: here, all 6,000 tickets. A **sample** is the part that you measure. The mean of a sample is an **estimate** of the true mean.

The next cell defines `sample_means(pool, n, k)`. It draws `k` samples of `n` tickets from `pool`, at random, and returns the mean of each sample. It picks tickets with replacement, as the explorer does. Then it draws 100 random samples of 30 tickets from the whole population.

> **Predict.** Will the 100 sample means be exactly 26.8? How far from 26.8 will most of them be? Then run the cell.

In [ ]:
true_mean = statistics.mean(population)

def sample_means(pool, n, k, seed=1):
    """Draw k samples of n tickets from pool, at random, and return their means."""
    random.seed(seed)
    return [statistics.mean(random.choices(pool, k=n)) for _ in range(k)]

means = sample_means(population, 30, 100)
print("first five sample means:", [round(m, 1) for m in means[:5]])
print("mean of the 100 sample means:", round(statistics.mean(means), 1))
print("SD of the 100 sample means:  ", round(statistics.stdev(means), 1))
print("true mean:                   ", round(true_mean, 1))

> **Check.** You should see the first five sample means 24.4, 31.0, 20.6, 23.1 and 27.5. No single sample is exact. The mean of the 100 sample means is 26.3, close to the true mean of 26.8, and the means typically scatter by about 3.1 minutes. **Random samples scatter around the true value.**

The next cell sorts the 100 sample means into 2-minute bins. Run it. You should see a hill from 22 to 30 minutes, with most means between 20 and 34.

In [ ]:
lefts = list(range(16, 40, 2))
counts = [len([m for m in means if left <= m < left + 2]) for left in lefts]
bars([f"{left}–{left + 2}" for left in lefts], counts, width=30)

## 2. A biased sample: only rated tickets

Now take the samples only from the rated tickets, as with the **Only rated tickets** switch in the explorer.

> **Predict.** Will the means of rated-only samples gather around 26.8 minutes, or somewhere else? Then run the cell.

In [ ]:
rated_means = sample_means(rated, 30, 100)
print("mean of 100 rated-only sample means:", round(statistics.mean(rated_means), 1))
print("true mean of all tickets:           ", round(true_mean, 1))
counts = [len([m for m in rated_means if left <= m < left + 2]) for left in lefts]
bars([f"{left}–{left + 2}" for left in lefts], counts, width=30)

> **Check.** You should see a mean of 20.6 minutes and a hill from 18 to 22 minutes, far from 26.8. The rated-only means scatter around the wrong value. That is **sampling bias**: the way the sample was chosen moves every estimate in the same direction.

Where does the bias come from? The next cell compares the shares of the population and of the rated tickets in 10-minute bins. Run it. You should see that 46.0% of the rated tickets got a reply after 10 to 20 minutes, against 33.3% of all tickets. Slow replies (40 minutes or more) are 18.2% of all tickets but only 6.0% of the rated ones.

In [ ]:
def shares(values):
    """The percentage of the values in each 10-minute bin, and 70 minutes or more."""
    rows = [100 * len([v for v in values if left <= v < left + 10]) / len(values) for left in range(0, 70, 10)]
    return rows + [100 * len([v for v in values if v >= 70]) / len(values)]

labels = [f"{left}–{left + 10}" for left in range(0, 70, 10)] + ["70+"]
table(["Minutes", "All tickets (%)", "Rated tickets (%)"],
      [[label, a, b] for label, a, b in zip(labels, shares(population), shares(rated))], digits=1)
print("40 minutes or more:", round(sum(shares(population)[4:]), 1), "% of all,", round(sum(shares(rated)[4:]), 1), "% of rated")

## 3. Guided practice: measure the bias

> **Your turn.** Replace each `...` with a calculation, then run the check cell.
>
> - `share_rated`: the percentage of the 6,000 tickets that a customer rated.
> - `rated_mean`: the mean of the rated tickets.
> - `bias`: the rated mean minus the true mean, in minutes. A negative number means "too low".

In [ ]:
share_rated = ...     # in percent, for example 50.0
rated_mean = ...
bias = ...

Run the check cell.

In [ ]:
expect("the share of rated tickets", share_rated, 35.92, tol=0.01)
expect("the mean of the rated tickets", rated_mean, 20.94, tol=0.01)
expect("the bias", bias, -5.91, tol=0.01)

A report from rated tickets says "customers wait about 21 minutes", when the true mean is about 27.

## 4. Independent variation: more tickets do not fix bias

Change one thing: the sample size. The next cell draws one random sample and one rated-only sample of each size, and prints the error of each mean: the sample mean minus the true mean.

> **Predict.** When the sample grows from 30 to 2,000 tickets, what happens to the error of the random sample? And to the error of the rated-only sample? Then run the cell.

In [ ]:
rows = []
for n in [30, 300, 2000]:
    random_mean = sample_means(population, n, 1, seed=3)[0]
    rated_only_mean = sample_means(rated, n, 1, seed=3)[0]
    rows.append([n, random_mean - true_mean, rated_only_mean - true_mean])
table(["Tickets n", "Error, random sample", "Error, rated only"], rows, digits=1)

> **Check.** You should see errors for the random sample of −4.1, 0.9 and 0.0 minutes: they become small. The errors for the rated-only sample are −6.9, −5.3 and −6.2 minutes: they stay at about −6. A larger sample makes chance smaller. It does not remove the bias, because every rated ticket comes from the same biased group.

> **Your turn: change one thing.** In the cell above, change the `seed=3` to another number, for example `seed=8`, and run it again. The numbers change. For the large samples, the pattern stays: the random error is close to 0, and the rated-only error is close to −6.

## 5. Linked tickets carry less information

A random sample also assumes that the tickets are **independent**. Suppose 60 of 300 test tickets come from one delivery outage and are almost the same. If the model sorts one of them correctly, it sorts all 60 correctly.

The next cell makes 2,000 made-up test sets of 300 tickets for a model that is right on 92% of tickets. In the first kind, every ticket is independent. In the second kind, 240 tickets are independent and 60 are copies of one problem: one coin decides all 60. The cell prints the middle 95% of the 2,000 accuracies.

> **Predict.** Both kinds have 300 tickets. Will their accuracies vary by the same amount? Then run the cell.

In [ ]:
def test_accuracy(linked):
    """The accuracy (%) on one made-up test set of 300 tickets."""
    if linked:
        correct = sum(random.random() < 0.92 for _ in range(240))
        correct += 60 * (random.random() < 0.92)    # one problem decides all 60 copies
    else:
        correct = sum(random.random() < 0.92 for _ in range(300))
    return 100 * correct / 300

random.seed(4)
for linked in [False, True]:
    accs = sorted(test_accuracy(linked) for _ in range(2000))
    name = "60 linked tickets" if linked else "all independent  "
    print(f"{name}: middle 95% from {accs[50]:.1f}% to {accs[1949]:.1f}%")

> **Check.** You should see 88.7% to 95.0% for independent tickets, and 73.3% to 96.0% for the test sets with 60 linked tickets. The test sets with linked tickets have the same size, but their result is much less certain. When the one outage problem goes wrong, 60 tickets go wrong together.

## 6. Failure case: data leakage from a ticket-by-ticket split

The lesson's failure case: the data was split ticket by ticket, so one customer's messages went to both training and test. The next cell has 12 made-up tickets from 5 customers. It splits them ticket by ticket: every third ticket goes to test. Run it.

> **Predict.** How many customers have tickets in both training and test?

In [ ]:
tickets = [
    ("T-5001", "C-11"), ("T-5002", "C-12"), ("T-5003", "C-11"), ("T-5004", "C-13"),
    ("T-5005", "C-14"), ("T-5006", "C-12"), ("T-5007", "C-15"), ("T-5008", "C-11"),
    ("T-5009", "C-13"), ("T-5010", "C-14"), ("T-5011", "C-15"), ("T-5012", "C-12"),
]   # (ticket, customer)

test = [t for i, t in enumerate(tickets) if i % 3 == 2]
train = [t for i, t in enumerate(tickets) if i % 3 != 2]
both = {c for _, c in test} & {c for _, c in train}
print("test tickets:", [t for t, _ in test])
print("customers in both training and test:", sorted(both))

> **Check.** You should see 4 test tickets and 3 customers in both: C-11, C-12 and C-13. The model can learn a customer's words from training and recognise them in test. The test result is then too high.

> **Your turn.** Split by **customer**: put all tickets of customers C-12 and C-15 in `test_by_customer`, and all other tickets in `train_by_customer`. Then run the check cell.

In [ ]:
test_customers = {"C-12", "C-15"}
test_by_customer = ...     # the tickets of the test customers
train_by_customer = ...    # all other tickets

Run the check cell.

In [ ]:
if test_by_customer is ... or train_by_customer is ...:
    print("Not yet: replace each ... with a list of tickets, then run the cell again.")
else:
    overlap = {c for _, c in test_by_customer} & {c for _, c in train_by_customer}
    expect("the number of test tickets", len(test_by_customer), 5)
    expect("the number of training tickets", len(train_by_customer), 7)
    expect("the number of customers in both", len(overlap), 0)

Now no customer is on both sides. In real work, also remove copies and near-copies across the split, and where possible test on the newest weeks (a time-based split).

## Check your understanding and recap

Answer the **knowledge checks** at the end of the [lesson page](https://learning.nextia-ai.com/courses/math/m05/samples-and-populations/). They count toward your certificate when you are signed in and enrolled. The notebook does not have them.

In this notebook, random samples of 30 tickets scattered around the true mean of 26.8 minutes. Rated-only samples scattered around 20.9 minutes, about 6 minutes too low, and larger samples did not fix that. Test sets with 60 linked tickets gave much less certain results than 300 independent tickets. A split by customer kept every customer on one side. The guided practice and the independent variation on the lesson page are about reading a test set; write those answers in `c03-notes.md`.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Estimate uncertainty](https://learning.nextia-ai.com/courses/math/m05/estimate-uncertainty/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/math/m05/samples-and-populations/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The Larkfield tickets are fictional and have no real people.